In [1]:
TRAIN = "train.csv"
TEST = "test.csv"
SOLUTION = "sample_solution.csv"

DB_FILE = "fuel_blend.db"
TABLE_PREFIX = "blends"   # will create blends_train, blends_test, blends_solution

In [14]:
import pandas as pd, sqlite3
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import warnings

# Suppress feature name warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [ ]:

conn = sqlite3.connect(DB_FILE)

# Load CSVs to SQL tables (overwrites if they exist)
pd.read_csv(TRAIN).to_sql(f"{TABLE_PREFIX}_train", conn, if_exists="replace", index=False)
pd.read_csv(TEST).to_sql(f"{TABLE_PREFIX}_test", conn, if_exists="replace", index=False)
pd.read_csv(SOLUTION).to_sql(f"{TABLE_PREFIX}_solution", conn, if_exists="replace", index=False)

# Read back from SQL
df_train = pd.read_sql_query(f"SELECT * FROM {TABLE_PREFIX}_train", conn)
df_test = pd.read_sql_query(f"SELECT * FROM {TABLE_PREFIX}_test", conn)
df_sol  = pd.read_sql_query(f"SELECT * FROM {TABLE_PREFIX}_solution", conn)
conn.close()

print(df_train.shape, df_test.shape, df_sol.shape)
df_train.head()


(2000, 65) (500, 56) (500, 11)


,Component1_fraction,Component2_fraction,Component3_fraction,Component4_fraction,Component5_fraction,Component1_Property1,Component2_Property1,Component3_Property1,Component4_Property1,Component5_Property1,...,BlendProperty1,BlendProperty2,BlendProperty3,BlendProperty4,BlendProperty5,BlendProperty6,BlendProperty7,BlendProperty8,BlendProperty9,BlendProperty10
0,0.21,0.00,0.42,0.25,0.12,-0.021782,1.981251,0.020036,0.140315,1.032029,...,0.489143,0.607589,0.321670,-1.236055,1.601132,1.384662,0.305850,0.193460,0.580374,-0.762738
1,0.02,0.33,0.19,0.46,0.00,-0.224339,1.148036,-1.107840,0.149533,-0.354000,...,-1.257481,-1.475283,-0.437385,-1.402911,0.147941,-1.143244,-0.439171,-1.379041,-1.280989,-0.503625
2,0.08,0.08,0.18,0.50,0.16,0.457763,0.242591,-0.922492,0.908213,0.972003,...,1.784349,0.450467,0.622687,1.375614,-0.428790,1.161616,0.601289,0.872950,0.660000,2.024576
3,0.25,0.42,0.00,0.07,0.26,-0.577734,-0.930826,0.815284,0.447514,0.455717,...,-0.066422,0.483730,-1.865442,-0.046295,-0.163820,-0.209693,-1.840566,0.300293,-0.351336,-1.551914
4,0.26,0.16,0.08,0.50,0.00,0.120415,0.666268,-0.626934,2.725357,0.392259,...,-0.118913,-1.172398,0.301785,-1.787407,-0.493361,-0.528049,0.286344,-0.265192,0.430513,0.735073


In [4]:
df_test.head()

,ID,Component1_fraction,Component2_fraction,Component3_fraction,Component4_fraction,Component5_fraction,Component1_Property1,Component2_Property1,Component3_Property1,Component4_Property1,...,Component1_Property9,Component2_Property9,Component3_Property9,Component4_Property9,Component5_Property9,Component1_Property10,Component2_Property10,Component3_Property10,Component4_Property10,Component5_Property10
0,1,0.18,0.05,0.32,0.37,0.08,-0.177804,-0.741219,0.769821,-0.877069,...,-0.265376,0.123432,0.028533,-0.173365,1.297923,0.323299,-0.315146,0.625518,-0.514342,-0.777057
1,2,0.00,0.50,0.00,0.37,0.13,2.501354,0.177344,-0.498739,-0.196742,...,-0.787677,-0.757905,-0.280561,-1.965970,0.543475,-0.906851,0.962341,-0.183757,0.310871,-1.329042
2,3,0.16,0.00,0.17,0.50,0.17,1.547324,0.891479,0.030627,-0.368678,...,-0.710026,-1.422693,0.874071,-1.016144,0.093525,1.048525,-1.321851,0.356640,-0.869543,-0.177255
3,4,0.50,0.00,0.17,0.16,0.17,-0.424427,1.016862,-1.182979,-0.854225,...,-0.551366,0.257105,-0.077337,-0.721031,-0.760365,-0.507690,1.346556,-0.001529,-1.008445,1.726105
4,5,0.00,0.00,0.50,0.50,0.00,-0.187062,-0.762173,-0.473660,2.074087,...,-1.811468,-0.181223,-0.475933,0.234775,-0.909020,1.238203,-1.805664,0.980417,-1.354932,-0.657513


In [15]:
X = df_train.drop(columns=df_train.columns[-10:])
y = df_train[df_train.columns[-10:]]

print("Feature shape:", X.shape, "Target shape:", y.shape)

Feature shape: (2000, 55) Target shape: (2000, 10)


In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
X_tr, X_va, y_tr, y_va = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [9]:
model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1
))
model.fit(X_tr, y_tr)


,estimator,RandomForestR...ndom_state=42)
,n_jobs,None
,n_estimators,300
,criterion,'squared_error'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0


In [16]:
from sklearn.metrics import mean_squared_error
y_pred = model.predict(X_va)

from sklearn.metrics import r2_score
r2 = r2_score(y_va, y_pred)

mse = mean_squared_error(y_va, y_pred)
rmse = np.sqrt(mse)

print(f"Validation R2 = {r2:.4f}, RMSE = {rmse:.4f}")

Validation R2 = 0.8659, RMSE = 0.3619


In [ ]:
# Ensure test data uses same feature columns as training
X_test = df_test[X.columns]  
X_test_scaled = scaler.transform(X_va)

y_test_pred = model.predict(X_test_scaled)

# Store predictions
df_pred = pd.DataFrame(y_test_pred, columns=y.columns)
df_pred.insert(0, "ID", df_test["ID"])   # keep ID to match sample_solution

df_pred.head()

,ID,BlendProperty1,BlendProperty2,BlendProperty3,BlendProperty4,BlendProperty5,BlendProperty6,BlendProperty7,BlendProperty8,BlendProperty9,BlendProperty10
0,1,-0.184330,0.123812,0.901448,-0.292926,2.803276,-0.752342,0.917727,-0.385088,-0.566093,0.969902
1,2,-0.688795,0.076815,1.219292,-0.923567,0.108092,-0.974990,1.313896,0.296002,-0.606769,0.116092
2,3,0.450218,-0.066403,1.330376,0.107053,2.253924,-0.362342,1.470503,0.384760,-0.174155,0.806871
3,4,-0.018028,0.689732,-1.259658,-0.104435,-0.640550,0.438233,-1.237692,-0.890099,1.731494,-1.371080
4,5,-1.018601,-1.966104,-0.898742,-0.825510,-0.090055,-1.970511,-0.879476,-1.111336,-1.451757,0.123552


In [12]:
conn = sqlite3.connect(DB_FILE)
df_pred.to_sql(f"{TABLE_PREFIX}_predictions", conn, if_exists="replace", index=False)
conn.close()